In [ ]:
# Ensure Internet toggle on the right side panel is switched ON!
!nvidia-smi

In [ ]:
%%bash
# 1. Clean up any corrupted directory versions cleanly
rm -rf /kaggle/working/spec-fastgs

# 2. Recursively clone using the correct branch name structure
git clone --recursive -b main https://github.com/0Nguyen0Cong0Tuan0/thesis-all.git /kaggle/working/spec-fastgs

# 3. Verify the layout structure
echo "📂 Verifying submodules directory contents:"
ls -la /kaggle/working/spec-fastgs/spec-fastgs/submodules/

In [ ]:
%%bash
# Create the datasets directory inside your workspace and copy files over from input mount
mkdir -p /kaggle/working/spec-fastgs/spec-fastgs/datasets
cp -r /kaggle/input/datasets/nctuan/spec-fastgs-datasets/datasets /kaggle/working/spec-fastgs/spec-fastgs/datasets/

In [ ]:
%%bash
# ============================================================
# R4 PREREQUISITE — generate monocular normal priors (v2.7, root cause B)
# IMPORTANT: run this BEFORE the conda-teardown cell below (the one that does
# `rm -rf /opt/conda` and installs torch 1.13.1). Here we still have Kaggle's
# stock MODERN torch, which Marigold (diffusers) needs. The output .npy maps are
# plain files and survive the rebuild. Skip this cell if you only want R3.
# ============================================================
set -e
cd /kaggle/working/spec-fastgs/spec-fastgs

# 1. Align the dataset layout NOW (same guard as the training cell; idempotent)
if [ -d "./datasets/datasets" ]; then
    echo "📦 Re-aligning dataset file structure..."
    mv ./datasets/datasets/* ./datasets/
    rm -rf ./datasets/datasets
fi
ls -d ./datasets/mipnerf360/counter/images >/dev/null && echo "✅ dataset ready"

# 2. Install the normal estimator into Kaggle's stock python (modern torch)
pip install -q -U diffusers transformers accelerate

# 3. Generate normals -> ./datasets/mipnerf360/counter/normals/<name>.npy (+ .png preview)
python tools/gen_normal_priors.py \
    -s ./datasets/mipnerf360/counter \
    -i images \
    --model marigold \
    --device cuda

echo "🧭 normal priors written:"
ls ./datasets/mipnerf360/counter/normals | head -5
echo "... total:" $(ls ./datasets/mipnerf360/counter/normals/*.npy | wc -l) "npy maps"


In [ ]:
%%bash
# Clean existing conda toolchain directories safely
rm -rf /opt/conda

# Reinstall isolated Miniconda (Python 3.10)
wget -q https://repo.anaconda.com/miniconda/Miniconda3-py310_23.11.0-1-Linux-x86_64.sh
bash Miniconda3-py310_23.11.0-1-Linux-x86_64.sh -b -p /opt/conda

# Activate custom installation
source /opt/conda/bin/activate

# Install compiler dependencies and CUDA Toolkit 11.7 matching constraints
/opt/conda/bin/conda install -y -c conda-forge cudatoolkit-dev=11.7 gcc_linux-64=11 gxx_linux-64=11

# Export local paths
export CUDA_HOME=/opt/conda
export PATH=$CUDA_HOME/bin:$PATH

echo "----- NVCC -----"
nvcc --version
echo "----- PTXAS -----"
ptxas --version

In [ ]:
%%bash
/opt/conda/bin/pip install torch==1.13.1+cu117 torchvision==0.14.1+cu117 \
  --index-url https://download.pytorch.org/whl/cu117

In [ ]:
%%bash
/opt/conda/bin/python - << 'EOF'
import torch
print("Torch version:", torch.__version__)
print("CUDA back-end:", torch.version.cuda)
print("GPU Available:", torch.cuda.is_available())
EOF

In [ ]:
%%bash
export CUDA_HOME=/opt/conda
export PATH=$CUDA_HOME/bin:$PATH
export LD_LIBRARY_PATH=$CUDA_HOME/lib:$LD_LIBRARY_PATH
export CC=gcc-11
export CXX=g++-11
export CUDAHOSTCXX=g++-11

BASE_DIR="/kaggle/working/spec-fastgs/spec-fastgs/submodules"

# 1. diff-gaussian-rasterization
cd "$BASE_DIR/diff-gaussian-rasterization_fastgs"
rm -rf build dist *.egg-info
/opt/conda/bin/pip install -v .

# 2. simple-knn
cd "../simple-knn"
rm -rf build dist *.egg-info
/opt/conda/bin/pip install -v .

# 3. fused-ssim
cd "../fused-ssim"
rm -rf build dist *.egg-info
/opt/conda/bin/pip install -v .

In [ ]:
%%bash
/opt/conda/bin/python - << 'EOF'
import diff_gaussian_rasterization_fastgs
import simple_knn
import fused_ssim
print("✅ FastGS CUDA extensions compiled and loaded successfully!")
EOF

In [ ]:
%%bash
export CUDA_HOME=/opt/conda
export PATH=$CUDA_HOME/bin:$PATH
export LD_LIBRARY_PATH=$CUDA_HOME/lib:$LD_LIBRARY_PATH
export CC=gcc-11
export CXX=g++-11
export CUDAHOSTCXX=g++-11

BASE_DIR="/kaggle/working/spec-fastgs/spec-fastgs/submodules"

cd "$BASE_DIR/diff-gaussian-rasterization_fastgs"
/opt/conda/bin/python setup.py bdist_wheel

cd "../simple-knn"
/opt/conda/bin/python setup.py bdist_wheel

cd "../fused-ssim"
/opt/conda/bin/python setup.py bdist_wheel

In [ ]:
%%bash
SRC="/kaggle/working/spec-fastgs/spec-fastgs/submodules"
DEST="/kaggle/working/fastgs_wheels_py310"

mkdir -p "$DEST"
find "$SRC" -name "*.whl" -exec cp {} "$DEST" \;

echo "✨ Wheels safely compiled and extracted to: $DEST"
ls -la "$DEST"

In [ ]:
%%bash
/opt/conda/bin/pip uninstall -y numpy
/opt/conda/bin/pip install "numpy<2" plyfile websockets tqdm

In [ ]:
%%bash
/opt/conda/bin/python -c "import fused_ssim; import diff_gaussian_rasterization_fastgs; import plyfile; print('🎉 All systems functional and ready for execution!')"

In [ ]:
%%bash
export PATH=/opt/conda/bin:$PATH
source /opt/conda/bin/activate
export CUDA_HOME=/opt/conda
export LD_LIBRARY_PATH=/opt/conda/lib:$LD_LIBRARY_PATH

cd /kaggle/working/spec-fastgs/spec-fastgs

# 1. Fix the double-nested dataset directory layout instantly
if [ -d "./datasets/datasets" ]; then
    echo "📦 Re-aligning dataset file structure..."
    mv ./datasets/datasets/* ./datasets/
    rm -rf ./datasets/datasets
fi

# 2. Fix your shell script's internal hardcoded paths dynamically
sed -i 's|/content/drive/MyDrive/spec-fastgs|/kaggle/working/spec-fastgs|g' run_spec-fastgs_big_r3.sh

# 3. Kick off training cleanly
bash run_spec-fastgs_big_r3.sh

In [ ]:
%%bash
# ============================================================
# RUN R4 (v2.7) — R3 residual specular loss + monocular normal prior
# Prerequisite: the "R4 PREREQUISITE" cell near the top must have produced
#   ./datasets/mipnerf360/counter/normals/*.npy
# Run this INSTEAD of the R3 cell above (not both). Outputs -> ./output/counter_r4
# ============================================================
export PATH=/opt/conda/bin:$PATH
source /opt/conda/bin/activate
export CUDA_HOME=/opt/conda
export LD_LIBRARY_PATH=/opt/conda/lib:$LD_LIBRARY_PATH

cd /kaggle/working/spec-fastgs/spec-fastgs

# 1. Dataset layout guard (idempotent — already aligned by the prereq cell)
if [ -d "./datasets/datasets" ]; then
    echo "📦 Re-aligning dataset file structure..."
    mv ./datasets/datasets/* ./datasets/
    rm -rf ./datasets/datasets
fi

# 2. Safety check: confirm the normal priors exist before training
if ! ls ./datasets/mipnerf360/counter/normals/*.npy >/dev/null 2>&1; then
    echo "❌ No normal priors found — run the 'R4 PREREQUISITE' cell first."
    exit 1
fi

# 3. Kick off R4. Watch the banner for 'code: v2.7-...' and the one-time line
#    '[normal-prior] alignment check ... mean cos(render,prior)=+0.xxx'.
#    If that cosine is strongly NEGATIVE, add  --normal_prior_flip  to
#    run_spec-fastgs_big_r4.sh (after '--normal_prior_dir normals') and rerun;
#    no need to regenerate the maps.
bash run_spec-fastgs_big_r4.sh


In [ ]:
import shutil
shutil.make_archive('spec_fastgs_output', 'zip', '/kaggle/working/spec-fastgs/spec-fastgs/output')